# Claude 스킬 소개

Claude의 스킬 기능으로 전문적인 문서를 만들고, 데이터를 분석하고, Excel·PowerPoint·PDF 생성으로 업무를 자동화하는 방법을 배웁니다.

> **실제로 보기:** 여기서 배울 스킬이 Claude의 파일 생성 기능을 구동합니다! **[Claude Creates Files](https://www.anthropic.com/news/create-files)**에서 이 스킬들이 Claude.ai에서 문서를 직접 만들고 편집하게 해 주는 모습을 확인해 보세요.

## 목차

1. [준비와 설치](#setup)
2. [스킬 이해하기](#understanding)
3. [사용 가능한 스킬 살펴보기](#discovering)
4. [빠른 시작: Excel](#excel-quickstart)
5. [빠른 시작: PowerPoint](#powerpoint-quickstart)
6. [빠른 시작: PDF](#pdf-quickstart)
7. [문제 해결](#troubleshooting)

## 1. 준비와 설치 {#setup}

### 사전 준비

시작하기 전에 다음을 확인하세요.
- Python 3.8 이상
- [console.anthropic.com](https://console.anthropic.com/)에서 발급받은 Anthropic API 키

### 환경 준비 (최초 1회)

**아직 환경을 준비하지 않았다면** 다음 단계를 따르세요.

#### 1단계: 가상 환경 만들기

```bash
# Navigate to the skills directory
cd /path/to/claude-cookbooks/skills

# Create virtual environment
python -m venv venv

# Activate it
source venv/bin/activate  # On macOS/Linux
# OR
venv\Scripts\activate     # On Windows
```

#### 2단계: 의존성 설치

```bash
# With venv activated, install requirements
pip install -r requirements.txt
```

#### 3단계: VSCode/Jupyter에서 커널 선택

**VSCode에서:**
1. 이 노트북을 엽니다
2. 오른쪽 위의 커널 선택기를 클릭합니다(예: "Python 3.11.x")
3. "Python Environments..."를 선택합니다
4. `./venv/bin/python` 인터프리터를 고릅니다

**Jupyter에서:**
1. Kernel 메뉴 → Change Kernel
2. venv에 맞는 커널을 선택합니다

#### 4단계: API 키 설정

```bash
# Copy the example file
cp .env.example .env

# Edit .env and add your API key:
# ANTHROPIC_API_KEY=sk-ant-api03-...
```

### 설치 빠른 점검

환경이 제대로 준비되었는지 확인하려면 아래 셀을 실행하세요:

**위에 ❌ 또는 ⚠️ 경고가 보인다면** 계속하기 전에 준비 단계를 완료하세요.

**anthropic SDK 버전이 너무 낮다면(0.71.0 이상 필요):**
```bash
pip install anthropic>=0.71.0
```
그런 다음 새 버전을 반영하려면 **Jupyter 커널을 재시작**하세요.

---

### API 설정

이제 API 키를 불러와 클라이언트를 설정하겠습니다:

### API 설정

**⚠️ 중요**: skills 디렉터리에 `.env` 파일을 만드세요:

```bash
# Copy the example file
cp ../.env.example ../.env
```

그런 다음 `../.env`를 편집해 Anthropic API 키를 추가하세요.

In [ ]:
import os
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))


from anthropic import Anthropic
from dotenv import load_dotenv

# Import our file utilities
from file_utils import (
    download_all_files,
    extract_file_ids,
    get_file_info,
    print_download_summary,
)

# Load environment variables from parent directory
load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

if not API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY not found. Copy ../.env.example to ../.env and add your API key."
    )

# Initialize client
# Note: We'll add beta headers per-request when using Skills
client = Anthropic(api_key=API_KEY)

# Create outputs directory if it doesn't exist
OUTPUT_DIR = Path.cwd().parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print("\n📝 Note: Beta headers will be added per-request when using Skills")

### 연결 테스트

API 연결이 잘 되는지 확인해 보겠습니다:

In [ ]:
# Simple test to verify API connection
test_response = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "Say 'Connection successful!' if you can read this.",
        }
    ],
)

print("API Test Response:")
print(test_response.content[0].text)
print(
    f"\n✓ Token usage: {test_response.usage.input_tokens} in, {test_response.usage.output_tokens} out"
)

## 2. 스킬 이해하기 {#understanding}

### 스킬이란?

**스킬**은 특정 작업에 대한 전문 역량을 Claude에 부여하는, 지시문·실행 코드·리소스를 체계적으로 묶은 패키지입니다. Claude가 필요할 때 찾아 동적으로 불러올 수 있는 "전문성 패키지"라고 생각하면 됩니다.

📖 엔지니어링 블로그 글 [스킬로 에이전트를 현실 세계에 대비시키기](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)도 읽어 보세요

### 스킬이 중요한 이유

MCP(Model Context Protocol)와 도구를 배우고 나면 스킬이 왜 중요한지 궁금할 수 있습니다.

- **스킬은 개별 도구보다 상위 개념**입니다. 지시문, 코드, 리소스를 하나로 묶습니다
- **스킬은 조합 가능합니다**. 여러 스킬이 매끄럽게 함께 동작합니다
- **스킬은 효율적입니다**. 점진적 공개 덕분에 실제로 쓴 만큼만 비용을 냅니다
- **스킬에는 검증된 코드가 들어 있습니다**. 안정적으로 동작하는 헬퍼 스크립트가 시간을 아끼고 오류를 줄여 줍니다

### 주요 이점

- **전문가 수준의 성능**: 학습 곡선 없이 실무 수준의 결과를 얻습니다
- **검증된 헬퍼 스크립트**: 스킬에는 Claude가 곧바로 쓸 수 있는 검증된 동작 코드가 들어 있습니다
- **조직 지식**: 회사의 업무 흐름과 모범 사례를 패키지로 만듭니다
- **비용 효율**: 점진적 공개가 토큰 사용을 최소화합니다
- **신뢰성**: 미리 검증된 스크립트 덕분에 오류가 줄고 결과가 일관됩니다
- **시간 절약**: Claude가 코드를 처음부터 만드는 대신 기존 해법을 사용합니다
- **조합 가능**: 여러 스킬이 함께 복잡한 워크플로를 처리합니다

### 점진적 공개 구조

스킬은 3단계 적재 모델을 사용합니다.

![점진적 공개 — 스킬이 적재되는 방식](../assets/prog-disc-1.png)

1. **메타데이터**(name: 64자, description: 1024자): Claude가 스킬 이름과 설명을 봅니다
2. **전체 지시문**(5k 토큰 미만): 스킬이 관련 있을 때 적재됩니다
3. **연결된 파일**: 필요할 때만 추가 리소스가 적재됩니다

![점진적 공개 단계](../assets/prog-disc-2.png)

덕분에 작업은 효율적으로 유지하면서도 필요할 때 깊은 전문성을 제공할 수 있습니다. 처음에 Claude는 SKILL.md의 YAML 프런트매터에 담긴 메타데이터만 봅니다. 스킬이 관련 있을 때에만 헬퍼 스크립트와 리소스를 포함한 전체 내용을 적재합니다.

### 스킬 유형

| 유형 | 설명 | 예시 |
|------|-------------|----------|
| **Anthropic 관리형** | Anthropic이 관리하는 사전 제작 스킬 | `xlsx`, `pptx`, `pdf`, `docx` |
| **커스텀** | 특정 워크플로를 위해 사용자가 정의한 스킬 | 브랜드 가이드라인, 재무 모델 |

### 스킬 개념 개요

![스킬 개념 다이어그램](../assets/skills-conceptual-diagram.png)

이 다이어그램이 보여 주는 것:
- **스킬 디렉터리 구조**: SKILL.md와 보조 파일로 스킬이 어떻게 구성되는지
- **YAML 프런트매터**: Claude가 처음에 보는 메타데이터
- **점진적 적재**: 스킬이 어떻게 발견되고 필요할 때 적재되는지
- **조합 가능성**: 하나의 요청에서 여러 스킬이 함께 동작하는 모습

### 스킬과 코드 실행이 함께 동작하는 방식

스킬을 쓰려면 **코드 실행** 도구가 활성화되어 있어야 합니다. 일반적인 워크플로는 다음과 같습니다.

```python
# Use client.beta.messages.create() for Skills support
response = client.beta.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=4096,
    container={
        "skills": [
            {"type": "anthropic", "skill_id": "xlsx", "version": "latest"}
        ]
    },
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[{"role": "user", "content": "Create an Excel file..."}],
    # Use betas parameter instead of extra_headers
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"]
)
```

**무슨 일이 일어나나요:**
1. xlsx 스킬이 적재된 상태로 Claude가 요청을 받습니다
2. Claude가 코드 실행으로 파일을 만듭니다
3. 응답에 생성된 파일의 `file_id`가 담깁니다
4. **Files API**로 파일을 내려받습니다

**중요: 베타 API**
- `client.messages.create()`가 아니라 `client.beta.messages.create()`를 사용하세요
- `container` 파라미터는 베타 API에서만 사용할 수 있습니다
- 베타 기능을 켜려면 `betas` 파라미터를 사용하세요.
  - `code-execution-2025-08-25` — 코드 실행 활성화
  - `files-api-2025-04-14` — 파일 내려받기에 필요
  - `skills-2025-10-02` — 스킬 기능 활성화

⚠️ **참고**: 스킬을 사용할 때는 요청에 code_execution 도구를 반드시 포함해야 합니다.

### 토큰 사용 최적화

스킬은 프롬프트에 지시문을 넣는 방식에 비해 토큰 사용을 크게 줄여 줍니다.

| 방식 | 토큰 비용 | 성능 |
|----------|------------|-------------|
| 수동 지시문 | 요청당 5,000~10,000 토큰 | 품질이 들쭉날쭉 |
| 스킬(메타데이터만) | 최소(이름/설명만) | 전문가 수준 |
| 스킬(전체 적재) | 스킬을 쓸 때 약 5,000 토큰 | 전문가 수준 |

**가장 큰 이점:** 프롬프트를 부풀리지 않고도 여러 스킬을 담아 둘 수 있습니다. 실제로 쓰기 전까지 각 스킬은 메타데이터(이름 + 설명) 비용만 듭니다.

**예시**: 서식이 적용된 Excel 파일 만들기
- 스킬 없이: 모든 Excel 기능을 미리 설명하는 데 약 8,000 토큰
- 스킬 사용: 처음에는 최소한의 메타데이터 부담만, Excel 스킬이 호출될 때만 약 5,000 토큰
- **핵심**: 98% 절감은 초기 컨텍스트에 해당합니다. 스킬을 실제로 쓰면 전체 지시문이 적재됩니다.

**추가 이점:**
- 스킬에는 동작이 확인된 헬퍼 스크립트가 들어 있어 신뢰성이 높아집니다
- Claude가 코드를 처음부터 만드는 대신 검증된 패턴을 써서 시간을 아낍니다
- 더 일관되고 전문적인 결과를 얻습니다

### ⏱️ 예상 생성 시간

**⚠️ 중요**: 스킬로 문서를 생성하려면 코드 실행과 파일 생성이 필요해 시간이 걸립니다. 셀이 끝날 때까지 기다려 주세요.

**관찰된 생성 시간:**
- **Excel 파일**: 약 2분(차트와 서식 포함)
- **PowerPoint 발표 자료**: 약 1~2분(차트가 있는 간단한 2슬라이드)
- **PDF 문서**: 약 40~60초(간단한 문서)

**예상되는 동작:**
- 실행 중에는 셀에 `[*]`가 표시됩니다
- 1~2분 동안 "Executing..." 상태가 보일 수 있습니다
- **셀을 중단하지 마세요.** 끝까지 실행되게 두세요

**💡 권장 사항:**
1. **단순하게 시작하세요**: 최소 예제로 환경이 제대로 되었는지 확인하세요
2. **복잡도를 점차 높이세요**: 기능을 조금씩 추가하세요
3. **기다리세요**: 보통 40초에서 2분이 걸립니다
4. **참고**: 매우 복잡한 문서는 더 오래 걸릴 수 있으니 예제는 초점을 좁게 유지하세요

## 3. 사용 가능한 스킬 살펴보기 {#discovering}

### 내장 스킬 목록 보기

Anthropic이 관리하는 스킬로 어떤 것이 있는지 살펴보겠습니다:

In [ ]:
# List all available Anthropic skills
# Note: Skills API requires the skills beta header
client_with_skills_beta = Anthropic(
    api_key=API_KEY, default_headers={"anthropic-beta": "skills-2025-10-02"}
)

skills_response = client_with_skills_beta.beta.skills.list(source="anthropic")

print("Available Anthropic-Managed Skills:")
print("=" * 80)

for skill in skills_response.data:
    print(f"\n📦 Skill ID: {skill.id}")
    print(f"   Title: {skill.display_title}")
    print(f"   Latest Version: {skill.latest_version}")
    print(f"   Created: {skill.created_at}")

    # Get version details
    try:
        version_info = client_with_skills_beta.beta.skills.versions.retrieve(
            skill_id=skill.id, version=skill.latest_version
        )
        print(f"   Name: {version_info.name}")
        print(f"   Description: {version_info.description}")
    except Exception as e:
        print(f"   (Unable to fetch version details: {e})")

print(f"\n\n✓ Found {len(skills_response.data)} Anthropic-managed skills")

### 스킬 메타데이터 이해하기

각 스킬에는 다음이 있습니다.
- **skill_id**: 고유 식별자(예: "xlsx", "pptx")
- **version**: 버전 번호 또는 "latest"
- **name**: 사람이 읽을 수 있는 이름
- **description**: 스킬이 하는 일
- **directory**: 스킬의 폴더 구조

### 버전 관리 전략

- Anthropic 스킬에는 `"latest"`를 사용하세요(권장)
- Anthropic이 스킬을 자동으로 갱신합니다
- 프로덕션 안정성을 위해서는 특정 버전으로 고정하세요
- 커스텀 스킬은 버전에 에포크 타임스탬프를 사용합니다

### 예제: 월간 예산 스프레드시트

간단한 한 줄짜리 예제와 상세 요청, 두 가지로 시작하겠습니다.

#### 간단한 예제 (1~2줄)
먼저 최소한의 프롬프트로 스킬이 어떻게 동작하는지 보겠습니다:

```python
# Simple prompt - Skills handle the complexity
prompt = "Create a quarterly sales report Excel file with revenue data and a chart"
```

#### 상세 예제
더 세밀하게 제어하려면 구체적인 요구 사항을 제시할 수 있습니다.
- 수입과 지출 항목
- 합계 수식
- 기본 서식

### 예제: 월간 예산 스프레드시트

다음을 포함한 간단한 예산 스프레드시트를 만듭니다.
- 수입과 지출 항목
- 합계 수식
- 기본 서식

**⏱️ 참고**: Excel 생성은 보통 **1~2분** 걸립니다(차트와 서식 포함). 실행 중에는 셀에 `[*]`가 표시되니 기다려 주세요!

In [ ]:
# Create an Excel budget spreadsheet
excel_response = client.beta.messages.create(  # Note: Using beta.messages for Skills support
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "xlsx", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a monthly budget Excel spreadsheet with the following:

Income:
- Salary: $5,000
- Freelance: $1,200
- Investments: $300

Expenses:
- Rent: $1,500
- Utilities: $200
- Groceries: $600
- Transportation: $300
- Entertainment: $400
- Savings: $1,000

Include:
1. Formulas to calculate total income and total expenses
2. A formula for net savings (income - expenses)
3. Format currency values properly
4. Add a simple column chart showing income vs expenses
5. Use professional formatting with headers
""",
        }
    ],
    # Use betas parameter for beta features
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("Excel Response:")
print("=" * 80)
for content in excel_response.content:
    if content.type == "text":
        print(content.text)
    elif content.type == "tool_use":
        print(f"\n🔧 Tool: {content.name}")
        if hasattr(content, "input"):
            print(f"   Input preview: {str(content.input)[:200]}...")

print("\n\n📊 Token Usage:")
print(f"   Input: {excel_response.usage.input_tokens}")
print(f"   Output: {excel_response.usage.output_tokens}")

### Excel 파일 내려받기

이제 file_id를 추출해 생성된 Excel 파일을 내려받겠습니다:

In [ ]:
# Extract file IDs from the response
file_ids = extract_file_ids(excel_response)

if file_ids:
    print(f"✓ Found {len(file_ids)} file(s)\n")

    # Download all files
    results = download_all_files(
        client, excel_response, output_dir=str(OUTPUT_DIR), prefix="budget_"
    )

    # Print summary
    print_download_summary(results)

    # Show file details
    for file_id in file_ids:
        info = get_file_info(client, file_id)
        if info:
            print("\n📄 File Details:")
            print(f"   Filename: {info['filename']}")
            print(f"   Size: {info['size'] / 1024:.1f} KB")
            print(f"   Created: {info['created_at']}")
else:
    print("❌ No files found in response")
    print("\nDebug: Response content types:")
    for i, content in enumerate(excel_response.content):
        print(f"  {i}. {content.type}")

**✨ 방금 무슨 일이 있었나요?**

1. Claude가 `xlsx` 스킬로 전문적인 Excel 파일을 만들었습니다
2. 스킬이 Excel 고유의 서식과 수식을 모두 처리했습니다
3. 파일은 Claude의 코드 실행 환경에서 만들어졌습니다
4. 응답에서 `file_id`를 추출했습니다
5. Files API로 파일을 내려받았습니다
6. 파일이 `outputs/budget_*.xlsx`에 저장되었습니다

Excel에서 파일을 열어 결과를 확인해 보세요!

## 5. 빠른 시작: PowerPoint {#powerpoint-quickstart}

이제 `pptx` 스킬로 PowerPoint 발표 자료를 만들어 보겠습니다.

### 예제: 매출 발표 자료

#### 간단한 예제 (1줄)
```python
# Minimal prompt - let Skills handle the details
prompt = "Create an executive summary presentation with 3 slides about Q3 results"
```

#### 상세 예제
**참고**: 생성 시간을 줄이고 핵심 기능을 보여 주기 위해 일부러 단순하게(슬라이드 2장, 차트 1개) 유지했습니다.

### 예제: 간단한 매출 발표 자료

**참고**: 생성 시간을 줄이고 핵심 기능을 보여 주기 위해 일부러 단순하게 유지했습니다.

In [ ]:
# Create a PowerPoint presentation
pptx_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "pptx", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a simple 2-slide PowerPoint presentation:

Slide 1: Title slide
- Title: "Q3 2025 Results"
- Subtitle: "Acme Corporation"

Slide 2: Revenue Overview
- Title: "Quarterly Revenue"
- Add a simple column chart showing:
  - Q1: $12M
  - Q2: $13M
  - Q3: $14M

Use clean, professional formatting.
""",
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PowerPoint Response:")
print("=" * 80)
for content in pptx_response.content:
    if content.type == "text":
        print(content.text)

print("\n\n📊 Token Usage:")
print(f"   Input: {pptx_response.usage.input_tokens}")
print(f"   Output: {pptx_response.usage.output_tokens}")

### PowerPoint 파일 내려받기

In [ ]:
# Download the PowerPoint file
file_ids = extract_file_ids(pptx_response)

if file_ids:
    results = download_all_files(
        client, pptx_response, output_dir=str(OUTPUT_DIR), prefix="q3_review_"
    )

    print_download_summary(results)

    print("\n✅ Open the presentation in PowerPoint or Google Slides to view!")
else:
    print("❌ No files found in response")

**⏱️ 참고**: PDF 생성은 간단한 문서라도 보통 **1~2분** 걸립니다. 실행 중에는 셀에 `[*]`가 표시되니 기다려 주세요!

### 예제: PDF 문서

#### 간단한 예제 (1줄)
```python
# Quick PDF generation
prompt = "Create a professional invoice PDF for $500 consulting services"
```

#### 상세 예제: 영수증
**참고**: 서식이 깔끔하게 나오도록 일부러 단순하게 유지했습니다.

### 예제: 간단한 영수증

**참고**: 서식이 깔끔하게 나오도록 일부러 단순하게 유지했습니다.

In [ ]:
# Create a PDF receipt
pdf_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "pdf", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a simple receipt PDF:

RECEIPT

Acme Corporation
Date: January 15, 2025
Receipt #: RCT-2025-001

Customer: Jane Smith

Items:
- Product A: $50.00
- Product B: $75.00
- Product C: $25.00

Subtotal: $150.00
Tax (8%): $12.00
Total: $162.00

Thank you for your business!

Use simple, clean formatting with clear sections.
""",
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PDF Response:")
print("=" * 80)
for content in pdf_response.content:
    if content.type == "text":
        print(content.text)

print("\n\n📊 Token Usage:")
print(f"   Input: {pdf_response.usage.input_tokens}")
print(f"   Output: {pdf_response.usage.output_tokens}")

### PDF 내려받고 확인하기

In [ ]:
# Download the PDF file
file_ids = extract_file_ids(pdf_response)

if file_ids:
    results = download_all_files(
        client, pdf_response, output_dir=str(OUTPUT_DIR), prefix="receipt_"
    )

    print_download_summary(results)

    # Verify PDF integrity
    for result in results:
        if result["success"]:
            file_path = result["output_path"]
            file_size = result["size"]

            # Basic PDF validation
            with open(file_path, "rb") as f:
                header = f.read(5)
                if header == b"%PDF-":
                    print(f"\n✅ PDF file is valid: {file_path}")
                    print(f"   File size: {file_size / 1024:.1f} KB")
                else:
                    print(f"\n⚠️ File may not be a valid PDF: {file_path}")
else:
    print("❌ No files found in response")

## 7. 문제 해결 {#troubleshooting}

### 흔한 문제와 해결 방법

### 문제 1: API 키를 찾을 수 없음

**오류:**
```
ValueError: ANTHROPIC_API_KEY not found
```

**해결:**
1. 상위 디렉터리에 `.env` 파일이 있는지 확인하세요
2. `ANTHROPIC_API_KEY=sk-ant-api03-...`이 설정되어 있는지 확인하세요
3. `.env`를 만들거나 수정한 뒤 Jupyter 커널을 재시작하세요

### 문제 2: container 파라미터를 인식하지 못함

**오류:**
```
TypeError: Messages.create() got an unexpected keyword argument 'container'
```

**해결:**
`client.messages.create()` 대신 `client.beta.messages.create()`를 사용하세요:
```python
# ✅ Correct - use beta.messages
response = client.beta.messages.create(
    model=MODEL,
    container={"skills": [...]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[...],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"]
)

# ❌ Incorrect - regular messages doesn't support container
response = client.messages.create(
    model=MODEL,
    container={"skills": [...]},  # Error!
    messages=[...]
)
```

### 문제 3: 스킬 베타에는 코드 실행 도구가 필요함

**오류:**
```
BadRequestError: Skills beta requires the code_execution tool to be included in the request.
```

**해결:**
스킬을 사용할 때는 code_execution 도구를 반드시 포함해야 합니다:
```python
# ✅ Correct
response = client.beta.messages.create(
    model=MODEL,
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[...],
    betas=["...", "skills-2025-10-02"]
)

# ❌ Incorrect - missing code_execution tool
response = client.beta.messages.create(
    model=MODEL,
    messages=[...],
    betas=["...", "skills-2025-10-02"]
)
```

### 문제 4: 응답에 파일이 없음

**오류:**
```
❌ No files found in response
```

**해결:**
1. 요청에 코드 실행 도구가 포함되어 있는지 확인하세요
2. 스킬이 적재되었는지 확인하세요(응답 내용 확인)
3. 해당 작업이 실제로 파일 생성을 필요로 하는지 확인하세요
4. 응답 텍스트에 오류 메시지가 있는지 살펴보세요

### 문제 5: 파일 내려받기 실패

**오류:**
```
Error retrieving file: File not found
```

**해결:**
1. 파일은 Anthropic 서버에서 보관 기간이 제한될 수 있습니다
2. 생성 직후에 파일을 내려받으세요
3. 응답에서 file_id를 제대로 추출했는지 확인하세요
4. betas 목록에 Files API 베타가 포함되어 있는지 확인하세요

### 토큰 최적화 팁

1. Anthropic 스킬에는 **"latest" 버전을 사용하세요** — 자동으로 최적화됩니다
2. **작업을 묶으세요** — 가능하면 한 대화에서 여러 파일을 만드세요
3. **컨테이너를 재사용하세요** — 이전 응답의 `container.id`를 써서 스킬 재적재를 피하세요
4. **구체적으로 지시하세요** — 명확한 지시는 반복 횟수를 줄여 줍니다

### API 요청 한도

요청 한도에 걸린다면:
- 재시도에 지수 백오프를 적용하세요
- 여러 파일은 배치로 처리하세요
- 한도를 높이려면 API 등급 상향을 고려하세요

## 다음 단계

🎉 **축하합니다!** Claude 스킬의 기초를 배웠습니다.

### 스킬이 실제로 쓰이는 모습 보기

이 스킬들이 Claude의 파일 생성 기능을 어떻게 구동하는지 공식 발표에서 확인해 보세요.
- **[Claude Creates Files](https://www.anthropic.com/news/create-files)** — 스킬이 Claude로 하여금 Excel, PowerPoint, PDF 파일을 직접 만들고 편집하게 해 주는 모습

### 학습 이어 가기

- **[노트북 2: 금융 업무 활용](02_skills_financial_applications.ipynb)** — 금융 데이터를 다루는 실제 비즈니스 사용 사례
- **[노트북 3: 커스텀 스킬 개발](03_skills_custom_development.ipynb)** — 여러분만의 특화된 스킬 만들기

### 지원 문서

- 📚 **[스킬로 Claude에게 여러분의 업무 방식 가르치기](https://support.claude.com/en/articles/12580051-teach-claude-your-way-of-working-using-skills)** — 스킬 사용자 가이드
- 🛠️ **[대화로 Claude와 함께 스킬 만드는 방법](https://support.claude.com/en/articles/12599426-how-to-create-a-skill-with-claude-through-conversation)** — 대화형 스킬 제작 가이드

### 참고 자료

- [Claude API 문서](https://docs.anthropic.com/en/api/messages)
- [스킬 문서](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)
- [스킬 모범 사례](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [Files API 문서](https://docs.claude.com/en/api/files-content)
- [Claude 지원](https://support.claude.com)

### 이런 실험을 해 보세요

1. 한 줄짜리 간단한 프롬프트로 스킬이 동작하는 모습을 확인해 보세요
2. 예산 예제를 수정해 항목을 더 넣어 보세요
3. 여러분의 데이터로 발표 자료를 만들어 보세요
4. 텍스트와 표를 결합한 PDF 보고서를 생성해 보세요
5. 하나의 요청에서 여러 스킬을 함께 사용해 보세요